## Langfuse Adapter Demo 

This notebook demonstrates how to use the generic **HallucinationEvaluator** with **EPRDetector** and **WEPRDetector** to automatically score LLM traces in Langfuse.

### Prerequisites

**1. Langfuse project**

Create a free project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to **Settings → API Keys** and copy your keys.

Create a `.env.demo` file at the project root.

**2. Local LLM server**

This demo runs inference locally using `llama.cpp`. 

* Install ``llama.cpp`` using brew, nix or winget

Then start the server by running :
* ``llama-server -hf unsloth/SmolLM2-135M-Instruct-GGUF --port 8080``

For more info, see [github.com/ggml-org/llama.cpp](https://github.com/ggml-org/llama.cpp)

### Run a generation and send it to Langfuse

We use the `@observe()` decorator to automatically trace the LLM call and send it to Langfuse.

In [12]:
from dotenv import load_dotenv
from langfuse.openai import OpenAI
from langfuse import observe, get_client

load_dotenv(".env.demo")

client = OpenAI(
    base_url="http://localhost:8080",
    api_key="test"
)

@observe()
def run_generation():
    completion = client.chat.completions.create(
        model="unsloth/SmolLM2-135M-Instruct-GGUF",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"},
        ],
        logprobs = True,
        top_logprobs = 5
    )
    return completion

completion = run_generation()
message = completion.choices[0].message.content
logprobs = completion.choices[0].logprobs

langfuse = get_client()
langfuse.flush()

### Score traces with EPR


In [13]:
from artefactual.adapters.hallucination_detector import EPRDetector
from artefactual.adapters.langfuse_evaluator import HallucinationEvaluator

evaluator = HallucinationEvaluator(
    name="EPR",
    langfuse_client=langfuse,
    detector=EPRDetector(),
)

traces = langfuse.api.trace.list(limit=2)

for trace in traces.data:
    score = evaluator.score_trace(trace.id)
    print(f"EPR Scored Trace : {trace.id} → {score}")

langfuse.flush()

EPR Scored Trace : ec42af1ff3c03e9d0c30b99f75797982 → 0.4298100173473358
EPR Scored Trace : 62c793237fda66a92cdd37a299e64845 → 0.29470959305763245


### Score traces with WEPR

In [14]:
from artefactual.adapters.hallucination_detector import WEPRDetector
from artefactual.adapters.langfuse_evaluator import HallucinationEvaluator

evaluator = HallucinationEvaluator(
    name="WEPR",
    langfuse_client=langfuse,
    detector=WEPRDetector(pretrained_model_name_or_path="mistralai/Mistral-Small-3.1-24B-Instruct-2503"),
)

traces = langfuse.api.trace.list(limit=2)

for trace in traces.data:
    score = evaluator.score_trace(trace.id)
    print(f"WEPR Scored Trace : {trace.id} → {score}")

langfuse.flush()

WEPR Scored Trace : ec42af1ff3c03e9d0c30b99f75797982 → 0.11702053993940353
WEPR Scored Trace : 62c793237fda66a92cdd37a299e64845 → 0.06846933811903
